# 🚀 Machine Learning Crash Course: Basics to Intermediate

Welcome to the **Comprehensive Machine Learning Crash Course**! This notebook covers essential and intermediate Machine Learning concepts, complete with mathematical intuition, scikit-learn code examples, interactive visualizations, and best practices.

--- 
### 📚 Table of Contents
1. [Module 1: Foundations of ML & Math Essentials](#module-1)
2. [Module 2: Data Preprocessing & Exploratory Data Analysis (EDA)](#module-2)
3. [Module 3: Supervised Learning - Regression](#module-3)
4. [Module 4: Supervised Learning - Classification](#module-4)
5. [Module 5: Unsupervised Learning](#module-5)
6. [Module 6: Model Validation, Tuning & Pipelines](#module-6)
7. [Module 7: Intermediate Topics & Neural Networks](#module-7)
8. [Summary & Next Steps](#summary)


## 🛠 Environment Setup & Essential Imports
Before diving in, let me ensure all required standard data science libraries are loaded.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-Learn Imports
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    mean_squared_error, r2_score, mean_absolute_error,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc, silhouette_score
)

# Plotting configuration
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
print("✅ All dependencies successfully imported!")


<a id='module-1'></a>
# 📐 Module 1: Foundations of Machine Learning & Math Essentials

### 1.1 What is Machine Learning?
Traditional programming involves writing explicit rules to process data and produce outputs. In contrast, **Machine Learning (ML)** uses algorithms to discover patterns from input data $X$ and labels $y$, allowing the computer to learn rules automatically.

- **Supervised Learning**: Learning from labeled data $(X, y)$. (e.g., House Price Prediction, Spam Detection)
- **Unsupervised Learning**: Discovering hidden patterns in unlabeled data $X$. (e.g., Customer Segmentation, Anomaly Detection)
- **Reinforcement Learning**: Learning optimal actions through trial-and-error rewards/penalties in an environment.

### 1.2 Mathematical Foundations
1. **Dot Product**: $\mathbf{a} \cdot \mathbf{b} = \sum_{i=1}^{n} a_i b_i = \mathbf{a}^T \mathbf{b}$
2. **Hypothesis Function (Linear)**: $\hat{y} = h_\theta(x) = \theta_0 + \theta_1 x_1 + \dots + \theta_n x_n = \mathbf{x}^T \boldsymbol{\theta}$
3. **Cost Function (Mean Squared Error)**: 
$$J(\boldsymbol{\theta}) = \frac{1}{2m} \sum_{i=1}^{m} \left( h_\theta(\mathbf{x}^{(i)}) - y^{(i)} \right)^2$$
4. **Gradient Descent Update Rule**:
$$\boldsymbol{\theta} := \boldsymbol{\theta} - \alpha \nabla J(\boldsymbol{\theta}) = \boldsymbol{\theta} - \alpha \frac{1}{m} \sum_{i=1}^{m} \left( h_\theta(\mathbf{x}^{(i)}) - y^{(i)} \right) \mathbf{x}^{(i)}$$
where $\alpha$ is the **learning rate**.


In [ ]:
# Implementation of Gradient Descent from scratch for a simple 1D function
# Minimize Loss Function: J(w) = (w - 4)^2 + 3

def cost_function(w):
    return (w - 4)**2 + 3

def compute_gradient(w):
    return 2 * (w - 4)

# Hyperparameters
w_initial = -2.0
learning_rate = 0.15
epochs = 25

history_w = [w_initial]
history_cost = [cost_function(w_initial)]

w = w_initial
for step in range(epochs):
    grad = compute_gradient(w)
    w = w - learning_rate * grad
    history_w.append(w)
    history_cost.append(cost_function(w))

print(f"Optimized weight: {w:.4f} (Optimal is 4.0)")

# Visualization of Gradient Descent Path
w_vals = np.linspace(-3, 11, 200)
plt.figure(figsize=(10, 5))
plt.plot(w_vals, cost_function(w_vals), 'b-', label='Cost Function $J(w)$')
plt.plot(history_w, history_cost, 'ro--', label='Gradient Descent Steps')
plt.title('Gradient Descent Optimization Visualization')
plt.xlabel('Weight (w)')
plt.ylabel('Cost J(w)')
plt.legend()
plt.show()


<a id='module-2'></a>
# 🧹 Module 2: Data Preprocessing & Exploratory Data Analysis (EDA)

Quality data is the prerequisite for effective machine learning ("Garbage in, garbage out").

### Key Preprocessing Steps:
1. **Imputation**: Handling missing values (Mean, Median, Mode).
2. **Categorical Encoding**: 
   - *One-Hot Encoding*: Nominal features without ordering (e.g., Colors).
   - *Ordinal/Label Encoding*: Features with natural ordering (e.g., Low < Medium < High).
3. **Feature Scaling**:
   - *Standardization (Z-score)*: $x' = \frac{x - \mu}{\sigma}$ (Mean = 0, Std = 1)
   - *MinMax Scaling*: $x' = \frac{x - x_{min}}{x_{max} - x_{min}}$ (Range $[0, 1]$)
4. **Data Leakage**: Preprocessing parameters (e.g., mean/std) **must** be calculated using training data only!


In [ ]:
# Generating a Synthetic Messy Dataset
np.random.seed(42)
n_samples = 200

data = pd.DataFrame({
    'Age': np.random.normal(35, 10, n_samples),
    'Income': np.random.exponential(50000, n_samples),
    'Department': np.random.choice(['Sales', 'Engineering', 'HR', 'Marketing'], n_samples),
    'YearsExperience': np.random.uniform(1, 15, n_samples),
    'Target_Salary': np.zeros(n_samples)
})

# Inject missing values
data.loc[data.sample(frac=0.08, random_state=42).index, 'Age'] = np.nan
data.loc[data.sample(frac=0.05, random_state=42).index, 'Income'] = np.nan

# Calculate target variable with noise
data['Target_Salary'] = 30000 + (data['YearsExperience'] * 4500) + (np.nan_to_num(data['Income']) * 0.15) + np.random.normal(0, 5000, n_samples)

print("Raw Data Head:")
display(data.head())
print("\nMissing values count:")
print(data.isnull().sum())


In [ ]:
# Preprocessing Pipeline with ColumnTransformer
X = data.drop(columns=['Target_Salary'])
y = data['Target_Salary']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

num_features = ['Age', 'Income', 'YearsExperience']
cat_features = ['Department']

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('encoder', OneHotEncoder(sparse_output=False, drop='first'))
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_features),
    ('cat', cat_pipeline, cat_features)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print(f"Processed Train Shape: {X_train_processed.shape}")
print(f"Processed Test Shape: {X_test_processed.shape}")


<a id='module-3'></a>
# 📈 Module 3: Supervised Learning - Regression

Regression models predict continuous quantitative target values.

### Key Concepts:
- **Bias-Variance Tradeoff**: 
  - *High Bias*: Underfitting (model is too simple).
  - *High Variance*: Overfitting (model fits training noise).
- **Polynomial Regression**: Adds feature powers $x^2, x^3$ to model non-linear relationships.
- **Regularization**:
  - **L1 Regularization (Lasso)**: Adds penalty $\alpha \sum |\theta_j|$. Performs feature selection by driving coefficients to 0.
  - **L2 Regularization (Ridge)**: Adds penalty $\alpha \sum \theta_j^2$. Shrinks coefficients to avoid extreme weights.

### Metrics:
- $\text{MSE} = \frac{1}{m} \sum (y - \hat{y})^2$
- $\text{RMSE} = \sqrt{\text{MSE}}$
- $\text{MAE} = \frac{1}{m} \sum |y - \hat{y}|$
- $R^2 = 1 - \frac{\sum (y - \hat{y})^2}{\sum (y - \bar{y})^2}$


In [ ]:
# Polynomial Regression & Regularization Comparison
np.random.seed(42)
X_synth = np.sort(np.random.rand(40, 1) * 6, axis=0)
y_synth = np.sin(X_synth).ravel() + np.random.normal(0, 0.2, 40)

X_plot = np.linspace(0, 6, 200)[:, np.newaxis]

# Compare Linear, High-degree Polynomial, and Regularized (Ridge) models
models = {
    'Underfit (Degree 1)': Pipeline([('poly', PolynomialFeatures(1)), ('reg', LinearRegression())]),
    'Overfit (Degree 10)': Pipeline([('poly', PolynomialFeatures(10)), ('reg', LinearRegression())]),
    'Balanced (Degree 10 + Ridge)': Pipeline([('poly', PolynomialFeatures(10)), ('reg', Ridge(alpha=1.0))])
}

plt.figure(figsize=(14, 5))
plt.scatter(X_synth, y_synth, color='black', label='Training Data', zorder=5)

for name, model in models.items():
    model.fit(X_synth, y_synth)
    y_plot = model.predict(X_plot)
    plt.plot(X_plot, y_plot, label=name, linewidth=2)

plt.ylim(-2, 2)
plt.title('Bias-Variance Tradeoff: Polynomial & Regularized Regression')
plt.xlabel('X')
plt.ylabel('y')
plt.legend()
plt.show()


<a id='module-4'></a>
# 🏷️ Module 4: Supervised Learning - Classification

Classification predicts categorical class labels.

### Key Algorithms:
1. **Logistic Regression**: Uses Sigmoid $\sigma(z) = \frac{1}{1 + e^{-z}}$ to output probabilities $[0, 1]$.
2. **K-Nearest Neighbors (KNN)**: Non-parametric, predicts based on majority vote of $k$ closest data points.
3. **Support Vector Machines (SVM)**: Finds optimal hyperplanes maximizing margin between classes.
4. **Decision Trees & Random Forests**: Tree-based splits minimizing Gini impurity or Entropy. Random Forests ensemble multiple trees for variance reduction.
5. **Gradient Boosting**: Sequentially builds weak trees, each correcting errors of previous trees.

### Evaluation Metrics:
- **Precision** = $\frac{TP}{TP + FP}$ (Quality of positive predictions)
- **Recall (Sensitivity)** = $\frac{TP}{TP + FN}$ (Coverage of positive cases)
- **F1-Score** = $2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$ (Harmonic mean)
- **ROC Curve & AUC**: Plots True Positive Rate vs False Positive Rate across thresholds.


In [ ]:
# Classification Algorithms Comparison & Decision Boundary Visualization
from sklearn.datasets import make_moons

X_m, y_m = make_moons(n_samples=300, noise=0.25, random_state=42)

classifiers = {
    'Logistic Regression': LogisticRegression(),
    'K-Nearest Neighbors (K=5)': KNeighborsClassifier(n_neighbors=5),
    'Support Vector Machine (RBF)': SVC(probability=True),
    'Random Forest': RandomForestClassifier(n_estimators=50, random_state=42),
    'Gradient Boosting': HistGradientBoostingClassifier(random_state=42)
}

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.ravel()

# Mesh grid for boundaries
x_min, x_max = X_m[:, 0].min() - 0.5, X_m[:, 0].max() + 0.5
y_min, y_max = X_m[:, 1].min() - 0.5, X_m[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02), np.arange(y_min, y_max, 0.02))

for idx, (name, clf) in enumerate(classifiers.items()):
    clf.fit(X_m, y_m)
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    
    axes[idx].contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.Spectral)
    axes[idx].scatter(X_m[:, 0], X_m[:, 1], c=y_m, cmap=plt.cm.Spectral, edgecolors='k', s=30)
    acc = clf.score(X_m, y_m)
    axes[idx].set_title(f"{name}\nAccuracy: {acc:.2f}")

# Hide empty subplot
axes[5].axis('off')
plt.tight_layout()
plt.show()


In [ ]:
# Detailed Evaluation: Confusion Matrix & ROC Curve
X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(X_m, y_m, test_size=0.3, random_state=42)
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_m, y_train_m)

y_pred_m = rf_model.predict(X_test_m)
y_probs_m = rf_model.predict_proba(X_test_m)[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Confusion Matrix
cm = confusion_matrix(y_test_m, y_pred_m)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title('Confusion Matrix')
axes[0].set_xlabel('Predicted Label')
axes[0].set_ylabel('True Label')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test_m, y_probs_m)
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
axes[1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
axes[1].set_title('Receiver Operating Characteristic (ROC) Curve')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend(loc="lower right")

plt.tight_layout()
plt.show()

print("Classification Report:")
print(classification_report(y_test_m, y_pred_m))


<a id='module-5'></a>
# 🔍 Module 5: Unsupervised Learning

Unsupervised algorithms find structure in data without explicit target labels $y$.

### 5.1 Clustering
- **K-Means Clustering**: Partition $N$ observations into $K$ clusters by minimizing within-cluster variance (inertia).
- **Elbow Method**: Plot Inertia vs $K$ to spot optimal cluster count.
- **Silhouette Coefficient**: Measures how similar an object is to its own cluster compared to other clusters (range $[-1, 1]$).

### 5.2 Dimensionality Reduction
- **Principal Component Analysis (PCA)**: Linear transformation finding orthogonal axes (Principal Components) that maximize dataset variance while reducing features.


In [ ]:
# K-Means Clustering & PCA Visualization
from sklearn.datasets import load_iris

iris = load_iris()
X_iris = iris.data
y_iris = iris.target

# 1. Find optimal K with Elbow Method & Silhouette
inertias = []
silhouette_scores = []
K_range = range(2, 8)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_iris)
    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_iris, labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(K_range, inertias, 'bo-')
axes[0].set_title('Elbow Method (Inertia vs K)')
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('Inertia')

axes[1].plot(K_range, silhouette_scores, 'ro-')
axes[1].set_title('Silhouette Score vs K')
axes[1].set_xlabel('Number of Clusters (K)')
axes[1].set_ylabel('Silhouette Score')
plt.tight_layout()
plt.show()

# 2. PCA 2D Reduction
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_iris)

plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y_iris, cmap='viridis', s=50)
plt.title(f'PCA 2D Projection of Iris Dataset\nExplained Variance Ratio: {pca.explained_variance_ratio_.sum():.2%}')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.colorbar(scatter, label='Class Label')
plt.show()


<a id='module-6'></a>
# ⚙️ Module 6: Model Validation, Tuning & Production Pipelines

Building production-grade ML models requires reliable cross-validation, systematic hyperparameter tuning, and leakage-proof pipelines.

### Key Principles:
1. **K-Fold Cross Validation**: Splits dataset into $K$ folds to ensure model generalizes beyond a single train-test split.
2. **Hyperparameter Optimization**:
   - **GridSearchCV**: Exhaustively searches over specified parameter values.
   - **RandomizedSearchCV**: Samples fixed number of parameter combinations randomly.
3. **Scikit-Learn Pipeline**: Encapsulates preprocessing and model training into a single reproducible object.


In [ ]:
# End-to-End Pipeline & Hyperparameter Tuning via GridSearchCV
from sklearn.datasets import load_breast_cancer

cancer = load_breast_cancer()
X_c, y_c = cancer.data, cancer.target

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_c, y_c, test_size=0.25, random_state=42)

# Create Complete Pipeline
full_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(random_state=42))
])

# Parameter Grid
param_grid = {
    'classifier__n_estimators': [50, 100, 150],
    'classifier__max_depth': [None, 5, 10],
    'classifier__min_samples_split': [2, 5]
}

grid_search = GridSearchCV(full_pipeline, param_grid, cv=5, scoring='f1', n_jobs=-1)
grid_search.fit(X_train_c, y_train_c)

print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best 5-Fold Cross-Validation F1-Score: {grid_search.best_score_:.4f}")

# Evaluate on unseen test set
test_pred = grid_search.predict(X_test_c)
print(f"Final Test Accuracy: {accuracy_score(y_test_c, test_pred):.4f}")


<a id='module-7'></a>
# 🧠 Module 7: Intermediate Topics & Neural Networks

### 7.1 Multi-Layer Perceptron (MLP)
An Artificial Neural Network consisting of an **Input Layer**, one or more **Hidden Layers**, and an **Output Layer**.

- **Forward Propagation**: $a^{(l)} = f(W^{(l)} a^{(l-1)} + b^{(l)})$, where $f$ is an activation function.
- **Activation Functions**:
  - **ReLU**: $f(z) = \max(0, z)$ (Solves vanishing gradient problem in hidden layers).
  - **Sigmoid**: $f(z) = \frac{1}{1 + e^{-z}}$ (Used for binary classification outputs).
  - **Softmax**: Normalizes logits into multi-class probability distribution.
- **Backpropagation**: Calculates gradients using the Chain Rule to adjust weights via Gradient Descent.

### 7.2 Feature Importance & Explainability
Understanding *why* a model makes predictions using feature attribution.


In [ ]:
# Multi-Layer Perceptron (MLP) Classifier & Feature Importance Analysis

# 1. Train MLP Neural Network
mlp = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=300, random_state=42)
mlp.fit(X_train_c, y_train_c)

plt.figure(figsize=(9, 4))
plt.plot(mlp.loss_curve_, color='purple', lw=2)
plt.title('MLP Neural Network Training Loss Curve')
plt.xlabel('Iterations')
plt.ylabel('Loss')
plt.show()

# 2. Feature Importance Visualization using Best Random Forest Model
best_rf = grid_search.best_estimator_['classifier']
importances = best_rf.feature_importances_
indices = np.argsort(importances)[::-1][:10]  # Top 10 features

plt.figure(figsize=(10, 5))
plt.title('Top 10 Most Important Features (Random Forest)')
plt.barh(range(10), importances[indices][::-1], align='center', color='teal')
plt.yticks(range(10), [cancer.feature_names[i] for i in indices][::-1])
plt.xlabel('Relative Feature Importance')
plt.show()


<a id='summary'></a>
# 🎯 Summary & Next Steps

Congratulations! You have completed the **Machine Learning Crash Course**.

### 📋 Machine Learning Workflow Checklist:
1. ✅ **Problem Formulation**: Define target, identify Supervised vs Unsupervised task.
2. ✅ **EDA & Cleaning**: Handle missing values, scale features, encode categoricals.
3. ✅ **Baseline Modeling**: Train simple linear/tree baseline models first.
4. ✅ **Cross-Validation**: Use K-Fold splits to prevent overfitting.
5. ✅ **Pipeline Construction**: Encapsulate preprocessing & model into `Pipeline` objects.
6. ✅ **Tuning & Ensembling**: Optimize hyperparameters using Grid Search or Boosting.
7. ✅ **Evaluation**: Evaluate on test set using metrics aligned with domain goals.

### 🚀 Recommended Next Step Roadmap:
- **Advanced Ensembles**: Dive into XGBoost, LightGBM, and CatBoost libraries.
- **Deep Learning**: Explore PyTorch or TensorFlow for Computer Vision (CNNs) & NLP (Transformers).
- **MLOps**: Learn MLflow, Docker, FastAPI for deploying models to production.
